# SWC Metadata Export

Export metadata for cells with SWC files to Excel, one tab per species.

In [12]:
import pandas as pd
import os
from pathlib import Path

In [13]:
# Define paths
swc_base_dir = Path('/root/capsule/scratch/20260807/swcs_to_publish')
output_file = swc_base_dir / 'swc_metadata.xlsx'

# Metadata file paths per species
metadata_files = {
    'human': '/root/capsule/data/human/metadata/20260807/human_metadata.csv',
    'macaque': '/root/capsule/data/macaque/metadata/20260722/macaque_metadata.csv',
    'mouse': '/root/capsule/data/mouse/metadata/20260826/mouse_metadata.csv'
}

# ID column name per species
id_columns = {
    'human': 'cell_id',
    'macaque': 'specimen_id',
    'mouse': 'specimen_id'
}

In [14]:
def get_specimen_ids_from_swc_dir(species_dir):
    """
    Get list of specimen/cell IDs that have SWC files in the directory.
    """
    if not species_dir.exists():
        return []
    
    specimen_ids = []
    for item in species_dir.iterdir():
        if item.is_dir() and item.name.isdigit():
            # Check if there's at least one SWC file in this directory
            if any(item.glob('*.swc')):
                specimen_ids.append(int(item.name))
    
    return sorted(specimen_ids)

In [15]:
# Get specimen IDs with SWC files for each species
species_order = ['human', 'macaque', 'mouse']

for species in species_order:
    species_dir = swc_base_dir / species
    specimen_ids = get_specimen_ids_from_swc_dir(species_dir)
    print(f"{species}: {len(specimen_ids)} cells with SWC files")
    print(f"  First few IDs: {specimen_ids[:5]}")

human: 368 cells with SWC files
  First few IDs: [539733464, 539736965, 539736968, 541549258, 541557114]
macaque: 51 cells with SWC files
  First few IDs: [1013982017, 1013982100, 1039496506, 1050914924, 1062712144]
mouse: 23 cells with SWC files
  First few IDs: [834776911, 849609230, 849620207, 850800876, 850830421]


In [16]:
# Load metadata and filter for cells with SWCs
species_dataframes = {}

for species in species_order:
    # Get specimen IDs with SWC files
    species_dir = swc_base_dir / species
    specimen_ids_with_swc = get_specimen_ids_from_swc_dir(species_dir)
    
    # Load metadata
    metadata_file = metadata_files[species]
    df = pd.read_csv(metadata_file)
    
    # Get the ID column name for this species
    id_col = id_columns[species]
    
    # Filter to only cells with SWC files
    df_filtered = df[df[id_col].isin(specimen_ids_with_swc)].copy()
    
    # Sort by ID and move ID column to first position
    df_filtered = df_filtered.sort_values(id_col)
    cols = [id_col] + [col for col in df_filtered.columns if col != id_col]
    df_filtered = df_filtered[cols]
    
    species_dataframes[species] = df_filtered
    
    print(f"\n{species.capitalize()}:")
    print(f"  Total cells in metadata: {len(df)}")
    print(f"  Cells with SWC files: {len(df_filtered)}")
    print(f"  ID column: {id_col}")


Human:
  Total cells in metadata: 1527
  Cells with SWC files: 368
  ID column: cell_id

Macaque:
  Total cells in metadata: 51
  Cells with SWC files: 51
  ID column: specimen_id

Mouse:
  Total cells in metadata: 29
  Cells with SWC files: 23
  ID column: specimen_id


In [17]:
# Write to Excel with one sheet per species
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for species in species_order:
        df = species_dataframes[species]
        sheet_name = species.capitalize()
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"Wrote {len(df)} rows to sheet '{sheet_name}'")

print(f"\nExcel file saved to: {output_file}")

Wrote 368 rows to sheet 'Human'
Wrote 51 rows to sheet 'Macaque'
Wrote 23 rows to sheet 'Mouse'

Excel file saved to: /root/capsule/scratch/20260807/swcs_to_publish/swc_metadata.xlsx


In [18]:
species_dataframes['mouse']

,specimen_id,Morpho?,restricted analysis?,subclass
22,834776911,True,True,L5_ET
14,849609230,True,True,L2-3_IT
8,849620207,True,True,L5_IT
25,850800876,True,True,L4_IT
19,850830421,True,True,L2-3_IT
9,851823699,True,True,L5_IT
26,851833032,True,True,L4_IT
12,852302327,True,True,L4_IT
6,853898242,True,True,L5_IT
11,894209801,True,True,L5_IT
